<a href="https://colab.research.google.com/github/sved07/XGB-Cascade-Classifier/blob/main/idk_cascade_full_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IDK Cascade Meta-Router Study: XGBoost vs. Random Forest vs. Logistic Regression vs. Decision Tree

This notebook measures whether the **choice of meta-router** in an IDK (I-Don't-Know) cascade meaningfully affects end-to-end latency and accuracy -- and whether router overhead itself, if left unmeasured, can silently erase the benefit of cascading altogether.

Compared router families: **XGBoost**, **Random Forest**, **Logistic Regression**, and a single **Decision Tree** -- across **CIFAR-10** and **CIFAR-100**, using a ResNet20/ResNet56 (Model A / Model C) cascade pair, over 8 random seeds per dataset.

In [1]:
!pip install -q xgboost scikit-learn pandas numpy torch torchvision scipy matplotlib


In [2]:
import time
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from scipy import stats
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using execution device: {device}")


Using execution device: cuda


## 1. Dataset and Models Setup


In [3]:
DATASETS = {
    'CIFAR-100': {
        'transform': transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
        ]),
        'num_classes': 100
    },
    'CIFAR-10': {
        'transform': transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2471, 0.2435, 0.2616])
        ]),
        'num_classes': 10
    }
}

def get_dataset(dataset_name, transform):
    if dataset_name == 'CIFAR-100':
        return torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
    elif dataset_name == 'CIFAR-10':
        return torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

def load_models_for_dataset(dataset_name):
    if dataset_name == 'CIFAR-100':
        model_a = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar100_resnet20", pretrained=True).to(device)
        model_b = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar100_resnet32", pretrained=True).to(device)
        model_c = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar100_resnet56", pretrained=True).to(device)
    elif dataset_name == 'CIFAR-10':
        model_a = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=True).to(device)
        model_b = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet32", pretrained=True).to(device)
        model_c = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet56", pretrained=True).to(device)
    model_a.eval()
    model_b.eval()
    model_c.eval()
    return model_a, model_b, model_c

transform = DATASETS['CIFAR-100']['transform']
val_dataset = get_dataset('CIFAR-100', transform)
model_a, model_b, model_c = load_models_for_dataset('CIFAR-100')
print("Base models and CIFAR-100 dataset loaded successfully.")


100%|██████████| 169M/169M [28:58<00:00, 97.2kB/s]


The repository chenyaofo_pytorch-cifar-models does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/resnet/cifar100_resnet20-23dac2f1.pt" to /root/.cache/torch/hub/checkpoints/cifar100_resnet20-23dac2f1.pt


100%|██████████| 1.11M/1.11M [00:00<00:00, 56.5MB/s]
Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/resnet/cifar100_resnet32-84213ce6.pt" to /root/.cache/torch/hub/checkpoints/cifar100_resnet32-84213ce6.pt


100%|██████████| 1.88M/1.88M [00:00<00:00, 80.5MB/s]
Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/resnet/cifar100_resnet56-f2eff4c8.pt" to /root/.cache/torch/hub/checkpoints/cifar100_resnet56-f2eff4c8.pt


100%|██████████| 3.41M/3.41M [00:00<00:00, 127MB/s]

Base models and CIFAR-100 dataset loaded successfully.


## 2. Single-Sample Latency Profiling


In [4]:
def measure_single_sample_latency(model, single_sample, n_reps=300, n_warmup=30, device=device):
    model.eval()
    times = []
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(single_sample)
        if device.type == "cuda":
            torch.cuda.synchronize()

        for _ in range(n_reps):
            start = time.perf_counter()
            _ = model(single_sample)
            if device.type == "cuda":
                torch.cuda.synchronize()
            end = time.perf_counter()
            times.append((end - start) * 1000)

    times = np.array(times)
    return times.mean(), times.std()

probe_batch = val_dataset[0][0].unsqueeze(0).to(device)
LATENCY_A, LATENCY_A_STD = measure_single_sample_latency(model_a, probe_batch)
LATENCY_B, LATENCY_B_STD = measure_single_sample_latency(model_b, probe_batch)
LATENCY_C, LATENCY_C_STD = measure_single_sample_latency(model_c, probe_batch)

print(f"Model A Single-Sample Latency: {LATENCY_A:.4f} +/- {LATENCY_A_STD:.4f} ms")
print(f"Model C Single-Sample Latency: {LATENCY_C:.4f} +/- {LATENCY_C_STD:.4f} ms")


Model A Single-Sample Latency: 2.4662 +/- 0.0348 ms
Model C Single-Sample Latency: 6.4395 +/- 0.3483 ms


## 3. Router Training Logic: XGBoost, Random Forest, Logistic Regression, Decision Tree

All four routers are trained on the same 3-feature telemetry (`confidence`, `entropy`, `margin`) extracted from Model A, with the same cost-sensitive class weighting so the comparison isn't confounded by one router being tuned more carefully than another. Logistic Regression and a shallow Decision Tree are included as the 'standard'/simple baselines requested alongside XGBoost and Random Forest.

In [5]:
EARLY_EXIT_THRESHOLD = 0.90

def extract_telemetry(loader, model_a, model_c, device=device):
    rows = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs_a = model_a(images)
            probs_a = torch.softmax(outputs_a, dim=1)
            conf_A, preds_a = torch.max(probs_a, dim=1)
            entropy_A = -torch.sum(probs_a * torch.log(probs_a + 1e-6), dim=1)
            top2, _ = torch.topk(probs_a, k=2, dim=1)
            margin_A = top2[:, 0] - top2[:, 1]

            outputs_c = model_c(images)
            _, preds_c = torch.max(outputs_c, dim=1)

            actual_route_to_c = ((preds_a != labels) & (preds_c == labels)).long()

            for i in range(images.size(0)):
                rows.append({
                    'confidence': conf_A[i].item(),
                    'entropy': entropy_A[i].item(),
                    'margin': margin_A[i].item(),
                    'correct_a': (preds_a[i] == labels[i]).item(),
                    'correct_c': (preds_c[i] == labels[i]).item(),
                    'target_route': actual_route_to_c[i].item(),
                })
    return pd.DataFrame(rows)

def _cost_sensitive_pos_weight(y_train, latency_c, cost_missed_fallback=1.0):
    asymmetric_multiplier = cost_missed_fallback / max(latency_c, 1e-6)
    num_pos = np.sum(y_train == 1)
    num_neg = np.sum(y_train == 0)
    base_weight = num_neg / max(num_pos, 1)
    return base_weight * asymmetric_multiplier

def train_xgb_router(X_train, y_train, latency_c, cost_missed_fallback=1.0, seed=42):
    pos_weight = _cost_sensitive_pos_weight(y_train, latency_c, cost_missed_fallback)
    router = xgb.XGBClassifier(
        n_estimators=50, max_depth=3, learning_rate=0.1,
        scale_pos_weight=pos_weight, eval_metric='logloss', random_state=seed
    )
    router.fit(X_train, y_train)
    return router

def train_rf_router(X_train, y_train, latency_c, cost_missed_fallback=1.0, seed=42):
    pos_weight = _cost_sensitive_pos_weight(y_train, latency_c, cost_missed_fallback)
    router = RandomForestClassifier(
        n_estimators=50, max_depth=3,
        class_weight={0: 1.0, 1: pos_weight}, random_state=seed
    )
    router.fit(X_train, y_train)
    return router

def train_logreg_router(X_train, y_train, latency_c, cost_missed_fallback=1.0, seed=42):
    """'Standard model' baseline -- the simplest common router choice."""
    pos_weight = _cost_sensitive_pos_weight(y_train, latency_c, cost_missed_fallback)
    router = LogisticRegression(
        class_weight={0: 1.0, 1: pos_weight}, max_iter=1000, random_state=seed
    )
    router.fit(X_train, y_train)
    return router

def train_dtree_router(X_train, y_train, latency_c, cost_missed_fallback=1.0, seed=42):
    """Single shallow tree -- isolates whether *ensembling* itself (bagging or
    boosting) is what costs overhead, independent of tree-count."""
    pos_weight = _cost_sensitive_pos_weight(y_train, latency_c, cost_missed_fallback)
    router = DecisionTreeClassifier(
        max_depth=3, class_weight={0: 1.0, 1: pos_weight}, random_state=seed
    )
    router.fit(X_train, y_train)
    return router

ROUTER_REGISTRY = {
    'xgb':    {'label': 'XGBoost',            'train_fn': train_xgb_router,    'color': 'tab:blue',   'marker': 'o'},
    'rf':     {'label': 'Random Forest',      'train_fn': train_rf_router,     'color': 'tab:orange', 'marker': '^'},
    'logreg': {'label': 'Logistic Regression','train_fn': train_logreg_router, 'color': 'tab:purple', 'marker': 'D'},
    'dtree':  {'label': 'Decision Tree',      'train_fn': train_dtree_router,  'color': 'tab:brown',  'marker': 'P'},
}


### 3.1 Meta-Heuristic Baseline: Genetic-Algorithm-Optimized Linear Skip Rule

The four routers above are all trained ML models, each with some inference-time
call into a library (`sklearn` or `xgboost`). This section asks a different
question: can a much simpler decision rule -- optimized via a genetic algorithm
rather than gradient descent or tree-splitting -- match their accuracy while
avoiding a model-library call at inference time entirely? The resulting rule is
a linear discriminant over the same three features (confidence, entropy,
margin); at inference time it costs one dot product and one comparison.

In [6]:
EARLY_EXIT_THRESHOLD = 0.90

def extract_telemetry(loader, model_a, model_c, device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')):
    rows = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs_a = model_a(images)
            probs_a = torch.softmax(outputs_a, dim=1)
            conf_A, preds_a = torch.max(probs_a, dim=1)
            entropy_A = -torch.sum(probs_a * torch.log(probs_a + 1e-6), dim=1)
            top2, _ = torch.topk(probs_a, k=2, dim=1)
            margin_A = top2[:, 0] - top2[:, 1]

            outputs_c = model_c(images)
            _, preds_c = torch.max(outputs_c, dim=1)

            actual_route_to_c = ((preds_a != labels) & (preds_c == labels)).long()

            for i in range(images.size(0)):
                rows.append({
                    'confidence': conf_A[i].item(),
                    'entropy': entropy_A[i].item(),
                    'margin': margin_A[i].item(),
                    'correct_a': (preds_a[i] == labels[i]).item(),
                    'correct_c': (preds_c[i] == labels[i]).item(),
                    'target_route': actual_route_to_c[i].item(),
                })
    return pd.DataFrame(rows)


def _cost_sensitive_pos_weight(y_train, latency_c, cost_missed_fallback=1.0):
    asymmetric_multiplier = cost_missed_fallback / max(latency_c, 1e-6)
    num_pos = np.sum(y_train == 1)
    num_neg = np.sum(y_train == 0)
    base_weight = num_neg / max(num_pos, 1)
    return base_weight * asymmetric_multiplier

torch.manual_seed(42)
subset = torch.utils.data.Subset(val_dataset, list(range(1000)))
loader = torch.utils.data.DataLoader(subset, batch_size=32, shuffle=False)
df_probe = extract_telemetry(loader, model_a, model_c)
X_probe = df_probe[['confidence', 'entropy', 'margin']]
y_probe = df_probe['target_route']
X_tr, X_te, y_tr, y_te = train_test_split(X_probe, y_probe, test_size=0.2, random_state=42)


def _weighted_accuracy_fitness(weights, X_arr, y_arr, sample_weights):
    score = X_arr @ weights[:3] + weights[3]
    preds = (score >= 0).astype(int)
    correct = (preds == y_arr).astype(float)
    return np.sum(correct * sample_weights) / np.sum(sample_weights)

def _genetic_algorithm_search(X_arr, y_arr, sample_weights, seed=42,
                                pop_size=40, generations=60, gene_range=8.0,
                                mutation_rate=0.2, mutation_scale=0.6):
    rng = np.random.default_rng(seed)
    population = rng.uniform(-gene_range, gene_range, size=(pop_size, 4))

    def fitness_of(ind):
        return _weighted_accuracy_fitness(ind, X_arr, y_arr, sample_weights)

    best_individual, best_fitness = None, -np.inf
    for _ in range(generations):
        fitnesses = np.array([fitness_of(ind) for ind in population])
        gen_best_idx = np.argmax(fitnesses)
        if fitnesses[gen_best_idx] > best_fitness:
            best_fitness = fitnesses[gen_best_idx]
            best_individual = population[gen_best_idx].copy()

        new_population = [best_individual.copy()]  # elitism
        while len(new_population) < pop_size:
            i, j = rng.integers(0, pop_size, size=2)
            parent1 = population[i] if fitnesses[i] > fitnesses[j] else population[j]
            i, j = rng.integers(0, pop_size, size=2)
            parent2 = population[i] if fitnesses[i] > fitnesses[j] else population[j]

            mask = rng.random(4) < 0.5
            child = np.where(mask, parent1, parent2)
            mutate_mask = rng.random(4) < mutation_rate
            child = child + mutate_mask * rng.normal(0, mutation_scale, size=4)
            child = np.clip(child, -gene_range, gene_range)
            new_population.append(child)

        population = np.array(new_population[:pop_size])

    return best_individual, best_fitness


class HeuristicRouter:
    """Linear skip rule: escalate iff w.features + bias >= 0. No model-library
    call at inference time -- just a dot product and a comparison."""
    def __init__(self, weights):
        self.weights = np.asarray(weights, dtype=float)

    def predict_proba(self, X):
        X_arr = X.values if hasattr(X, 'values') else np.asarray(X)
        score = X_arr @ self.weights[:3] + self.weights[3]
        prob_escalate = 1.0 / (1.0 + np.exp(-score))
        return np.column_stack([1 - prob_escalate, prob_escalate])


def train_heuristic_router(X_train, y_train, latency_c, cost_missed_fallback=1.0, seed=42):
    """Trains the linear skip rule via a genetic algorithm, optimizing the same
    cost-weighted classification objective as the other routers."""
    pos_weight = _cost_sensitive_pos_weight(y_train, latency_c, cost_missed_fallback)
    X_arr = X_train.values if hasattr(X_train, 'values') else np.asarray(X_train)
    y_arr = y_train.values if hasattr(y_train, 'values') else np.asarray(y_train)
    sample_weights = np.where(y_arr == 1, pos_weight, 1.0)

    best_weights, _ = _genetic_algorithm_search(X_arr, y_arr, sample_weights, seed=seed)
    return HeuristicRouter(best_weights)


ROUTER_REGISTRY['heuristic'] = {
    'label': 'GA Heuristic', 'train_fn': train_heuristic_router,
    'color': 'tab:green', 'marker': 'X',
}

_heuristic_bench = train_heuristic_router(X_tr, y_tr, LATENCY_C, seed=42)
_fit = _weighted_accuracy_fitness(_heuristic_bench.weights, X_tr.values, y_tr.values,
                                    np.where(y_tr.values == 1,
                                             _cost_sensitive_pos_weight(y_tr.values, LATENCY_C), 1.0))
print(f"GA Heuristic trained. Learned weights (conf, entropy, margin, bias): {_heuristic_bench.weights}")
print(f"Weighted training accuracy: {_fit:.4f}")
print(f"Registered in ROUTER_REGISTRY: {'heuristic' in ROUTER_REGISTRY}")

GA Heuristic trained. Learned weights (conf, entropy, margin, bias): [-2.21099823 -6.30625507 -8.          7.39036263]
Weighted training accuracy: 0.8677
Registered in ROUTER_REGISTRY: True


###3.2 Pure Threshold Baseline (No Machine Learning Router)
This section provides the performance of the cascade without using a machine learning router

In [ ]:
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49]
DATASET_NAMES = ['CIFAR-10', 'CIFAR-100']

# Discrete thresholds representing different latency/escalation budgets
TAU_SWEEP = [0.10, 0.30, 0.50, 0.70, 0.80, 0.85, 0.90, 0.95, 0.99]

sweep_records = []

for dname in DATASET_NAMES:
    transform = DATASETS[dname]['transform']
    dataset = get_dataset(dname, transform)
    m_a, m_b, m_c = load_models_for_dataset(dname)

    probe_img = dataset[0][0].unsqueeze(0).to(device)
    lat_a, _ = measure_single_sample_latency(m_a, probe_img)
    lat_b, _ = measure_single_sample_latency(m_b, probe_img)
    lat_c, _ = measure_single_sample_latency(m_c, probe_img)

    loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=False)
    df_full = extract_3stage_telemetry(loader, m_a, m_b, m_c)

    feats_a = ['conf_a', 'entropy_a', 'margin_a']
    feats_b = ['conf_b', 'entropy_b', 'margin_b']
    ren_a = {'conf_a': 'confidence', 'entropy_a': 'entropy', 'margin_a': 'margin'}
    ren_b = {'conf_b': 'confidence', 'entropy_b': 'entropy', 'margin_b': 'margin'}

    for seed in SEEDS:
        df_tr, df_te = train_test_split(df_full, test_size=0.2, random_state=seed)
        total_samples = len(df_te)

        # 1. Pure Threshold Sweeps (A -> C)
        for tau in TAU_SWEEP:
            esc_ac = (df_te['conf_a'] < tau).values
            acc_ac = np.mean(np.where(esc_ac, df_te['correct_c'], df_te['correct_a'])) * 100
            lat_ac = lat_a + (np.mean(esc_ac) * lat_c)
            sweep_records.append({
                'Method': f"Pure Threshold (A->C, τ={tau:.2f})",
                'Accuracy': acc_ac,
                'Latency': lat_ac
            })

        # 2. Pure Threshold Sweeps (A -> B -> C)
        for tau in TAU_SWEEP:
            res_3t = evaluate_3tier_threshold(df_te, tau1=tau, tau2=tau, lat_a=lat_a, lat_b=lat_b, lat_c=lat_c)
            sweep_records.append({
                'Method': f"Pure Threshold (A->B->C, τ={tau:.2f})",
                'Accuracy': res_3t['Accuracy'] * 100,
                'Latency': res_3t['Latency (ms)']
            })

        # 3. Reference: Pure Expert (Model C Only)
        sweep_records.append({
            'Method': 'Pure Expert (Model C Only)',
            'Accuracy': df_te['correct_c'].mean() * 100,
            'Latency': lat_c
        })

        # 4. Meta-Routers
        target_ac_tr = ((df_tr['correct_a'] == 0) & (df_tr['correct_c'] == 1)).astype(int)
        for r_key in ['xgb', 'dtree', 'heuristic']:
            meta = ROUTER_REGISTRY[r_key]
            train_fn = meta['train_fn']
            ovh = ROUTER_SINGLE_SAMPLE_OVERHEAD[r_key]

            # 2-Tier (A -> C)
            r_2t = train_fn(df_tr[feats_a].rename(columns=ren_a), target_ac_tr, lat_c, seed=seed)
            p_2t = r_2t.predict_proba(df_te[feats_a].rename(columns=ren_a))[:, 1]
            esc_2t = (p_2t >= 0.5)
            acc_2t = np.mean(np.where(esc_2t, df_te['correct_c'], df_te['correct_a'])) * 100
            lat_2t = lat_a + ovh + (np.mean(esc_2t) * lat_c)
            sweep_records.append({
                'Method': f"{meta['label']} (A->C)",
                'Accuracy': acc_2t,
                'Latency': lat_2t
            })

            # 3-Tier (A -> B -> C)
            r_ab = train_fn(df_tr[feats_a].rename(columns=ren_a), df_tr['target_ab'], lat_b, seed=seed)
            r_bc = train_fn(df_tr[feats_b].rename(columns=ren_b), df_tr['target_bc'], lat_c, seed=seed)
            p_ab = r_ab.predict_proba(df_te[feats_a].rename(columns=ren_a))[:, 1]
            p_bc = r_bc.predict_proba(df_te[feats_b].rename(columns=ren_b))[:, 1]

            esc_b = (p_ab >= 0.5)
            esc_c = esc_b & (p_bc >= 0.5)
            exit_b_mask = esc_b & (~(p_bc >= 0.5))
            exit_a_mask = ~esc_b

            acc_3t = np.mean(
                (df_te['correct_a'].values & exit_a_mask) +
                (df_te['correct_b'].values & exit_b_mask) +
                (df_te['correct_c'].values & esc_c)
            ) * 100
            lat_3t = lat_a + ovh + (np.mean(esc_b) * (lat_b + ovh)) + (np.mean(esc_c) * lat_c)
            sweep_records.append({
                'Method': f"{meta['label']} (A->B->C)",
                'Accuracy': acc_3t,
                'Latency': lat_3t
            })
df_sweep_all = pd.DataFrame(sweep_records)
summary_sweep = df_sweep_all.groupby('Method', sort=False).agg({
    'Accuracy': ['mean', 'std'],
    'Latency': ['mean', 'std']
})

print("=== Multi-Threshold Sweep Performance Across All Seeds & Datasets ===")
for m in summary_sweep.index:
    print(f"{m:<40} Accuracy : {summary_sweep.loc[m, ('Accuracy', 'mean')]:.2f} +/- {summary_sweep.loc[m, ('Accuracy', 'std')]:.2f} %")

print("\n=== End-to-End Cascade Latency ===")
for m in summary_sweep.index:
    print(f"{m:<40} Cascade Latency : {summary_sweep.loc[m, ('Latency', 'mean')]:.4f} +/- {summary_sweep.loc[m, ('Latency', 'std')]:.4f} ms")

 44%|████▍     | 75.7M/170M [12:52<17:44, 89.1kB/s]

## 4. Single-Sample Router Inference Micro-Overhead Timing

This isolates each router's *own* inference cost (the `predict_proba` call), separate from Model A/C latency. This overhead is what Section 4's `evaluate_router_cascade` folds into the end-to-end cascade latency -- without this step, the comparison silently ignores the router's own compute cost.

In [ ]:
def measure_router_overhead_per_sample(router, X_test, n_reps=5):
    X_arr = X_test.values if hasattr(X_test, 'values') else X_test
    times = []
    for _ in range(n_reps):
        for row in X_arr:
            row = row.reshape(1, -1)
            start = time.perf_counter()
            _ = router.predict_proba(row)
            end = time.perf_counter()
            times.append((end - start) * 1000)
    times = np.array(times)
    return times.mean(), times.std()

torch.manual_seed(42)
subset = torch.utils.data.Subset(val_dataset, list(range(1000)))
loader = torch.utils.data.DataLoader(subset, batch_size=32, shuffle=False)
df_probe = extract_telemetry(loader, model_a, model_c)
X_probe = df_probe[['confidence', 'entropy', 'margin']]
y_probe = df_probe['target_route']
X_tr, X_te, y_tr, y_te = train_test_split(X_probe, y_probe, test_size=0.2, random_state=42)

print("=== Router Overhead (isolated predict_proba cost) ===")
for name, spec in ROUTER_REGISTRY.items():
    bench_router = spec['train_fn'](X_tr, y_tr, LATENCY_C, seed=42)
    mean_oh, std_oh = measure_router_overhead_per_sample(bench_router, X_te.head(200))
    spec['demo_overhead_mean'] = mean_oh
    spec['demo_overhead_std'] = std_oh
    print(f"[METRIC] {spec['label']:<20s} Overhead : {mean_oh:.5f} +/- {std_oh:.5f} ms/sample")


## 5. 3-Tier Cascade Study: $A \rightarrow B \rightarrow C$ vs. $2\text{-Tier } A \rightarrow C$

This section evaluates whether introducing an intermediate model (ResNet-32 as Model B) provides a better accuracy-latency trade-off than skipping directly from ResNet-20 (Model A) to ResNet-56 (Model C), comparing both pure-threshold baselines and meta-routers.

###5.1 Telemetry Extraction for 3 Stages

In [ ]:
def extract_3stage_telemetry(loader, model_a, model_b, model_c, device=device):
    rows = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            # Model A
            out_a = model_a(images)
            probs_a = torch.softmax(out_a, dim=1)
            conf_a, preds_a = torch.max(probs_a, dim=1)
            entropy_a = -torch.sum(probs_a * torch.log(probs_a + 1e-6), dim=1)
            top2_a, _ = torch.topk(probs_a, k=2, dim=1)
            margin_a = top2_a[:, 0] - top2_a[:, 1]

            # Model B
            out_b = model_b(images)
            probs_b = torch.softmax(out_b, dim=1)
            conf_b, preds_b = torch.max(probs_b, dim=1)
            entropy_b = -torch.sum(probs_b * torch.log(probs_b + 1e-6), dim=1)
            top2_b, _ = torch.topk(probs_b, k=2, dim=1)
            margin_b = top2_b[:, 0] - top2_b[:, 1]

            # Model C
            out_c = model_c(images)
            _, preds_c = torch.max(out_c, dim=1)

            target_a_to_b = ((preds_a != labels) & (preds_b == labels)).long()
            target_b_to_c = ((preds_b != labels) & (preds_c == labels)).long()

            for i in range(images.size(0)):
                rows.append({
                    'conf_a': conf_a[i].item(), 'entropy_a': entropy_a[i].item(), 'margin_a': margin_a[i].item(),
                    'conf_b': conf_b[i].item(), 'entropy_b': entropy_b[i].item(), 'margin_b': margin_b[i].item(),
                    'correct_a': (preds_a[i] == labels[i]).item(),
                    'correct_b': (preds_b[i] == labels[i]).item(),
                    'correct_c': (preds_c[i] == labels[i]).item(),
                    'target_ab': target_a_to_b[i].item(),
                    'target_bc': target_b_to_c[i].item()
                })
    return pd.DataFrame(rows)

###5.2 Evaluatinon Helper Functions

In [ ]:
def evaluate_3tier_threshold(df_telemetry, tau1=0.85, tau2=0.90,
                             lat_a=LATENCY_A, lat_b=LATENCY_B, lat_c=LATENCY_C):
    total = len(df_telemetry)
    esc_to_b = (df_telemetry['conf_a'] < tau1).values
    exit_a = ~esc_to_b
    esc_to_c = esc_to_b & (df_telemetry['conf_b'] < tau2).values
    exit_b = esc_to_b & (~(df_telemetry['conf_b'] < tau2).values)

    correct = (
        (df_telemetry['correct_a'].values & exit_a).sum() +
        (df_telemetry['correct_b'].values & exit_b).sum() +
        (df_telemetry['correct_c'].values & esc_to_c).sum()
    )

    p_b = np.mean(esc_to_b)
    p_c = np.mean(esc_to_c)
    expected_latency = lat_a + (p_b * lat_b) + (p_c * lat_c)

    return {
        'Method': f'No Router (τ1={tau1}, τ2={tau2})',
        'Accuracy': correct / total,
        'Latency (ms)': expected_latency,
        'Escalated to B (%)': p_b * 100,
        'Escalated to C (%)': p_c * 100
    }

def evaluate_3tier_meta_router(router_type, df_train, df_test,
                               lat_a=LATENCY_A, lat_b=LATENCY_B, lat_c=LATENCY_C):
    meta = ROUTER_REGISTRY[router_type]
    train_fn = meta['train_fn']

    feats_a = ['conf_a', 'entropy_a', 'margin_a']
    feats_b = ['conf_b', 'entropy_b', 'margin_b']

    # Train stage routers
    router_ab = train_fn(df_train[feats_a].rename(columns={'conf_a':'confidence','entropy_a':'entropy','margin_a':'margin'}),
                         df_train['target_ab'], lat_b, seed=42)
    router_bc = train_fn(df_train[feats_b].rename(columns={'conf_b':'confidence','entropy_b':'entropy','margin_b':'margin'}),
                         df_train['target_bc'], lat_c, seed=42)

    # Measure per-sample router inference overhead
    t0 = time.perf_counter()
    p_ab = router_ab.predict_proba(df_test[feats_a].rename(columns={'conf_a':'confidence','entropy_a':'entropy','margin_a':'margin'}))[:, 1]
    ovh_ab = ((time.perf_counter() - t0) * 1000) / len(df_test)

    t0 = time.perf_counter()
    p_bc = router_bc.predict_proba(df_test[feats_b].rename(columns={'conf_b':'confidence','entropy_b':'entropy','margin_b':'margin'}))[:, 1]
    ovh_bc = ((time.perf_counter() - t0) * 1000) / len(df_test)

    esc_to_b = (p_ab >= 0.5)
    exit_a = ~esc_to_b
    esc_to_c = esc_to_b & (p_bc >= 0.5)
    exit_b = esc_to_b & (~(p_bc >= 0.5))

    total = len(df_test)
    correct = (
        (df_test['correct_a'].values & exit_a).sum() +
        (df_test['correct_b'].values & exit_b).sum() +
        (df_test['correct_c'].values & esc_to_c).sum()
    )

    rate_b = np.mean(esc_to_b)
    rate_c = np.mean(esc_to_c)
    expected_latency = lat_a + ovh_ab + (rate_b * (lat_b + ovh_bc)) + (rate_c * lat_c)

    return {
        'Method': f'With Router ({meta["label"]})',
        'Accuracy': correct / total,
        'Latency (ms)': expected_latency,
        'Escalated to B (%)': rate_b * 100,
        'Escalated to C (%)': rate_c * 100
    }

###5.3 Comparing the Two-Layer Cascade with the Three-Layer Cascade

In [ ]:
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49]
DATASET_NAMES = ['CIFAR-10', 'CIFAR-100']

# Profiled isolated single-sample router overheads (ms/sample)
ROUTER_SINGLE_SAMPLE_OVERHEAD = {
    'xgb': 0.3485,
    'rf': 3.2468,
    'logreg': 0.2018,
    'dtree': 0.1449,
    'heuristic': 0.0123
}

all_run_records = []

for dname in DATASET_NAMES:
    transform = DATASETS[dname]['transform']
    dataset = get_dataset(dname, transform)
    m_a, m_b, m_c = load_models_for_dataset(dname)

    # 1. Profile dataset-specific base model latencies
    probe_img = dataset[0][0].unsqueeze(0).to(device)
    lat_a, _ = measure_single_sample_latency(m_a, probe_img)
    lat_b, _ = measure_single_sample_latency(m_b, probe_img)
    lat_c, _ = measure_single_sample_latency(m_c, probe_img)

    # 2. Extract full dataset telemetry
    full_loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=False)
    df_telemetry_full = extract_3stage_telemetry(full_loader, m_a, m_b, m_c)

    feats_a = ['conf_a', 'entropy_a', 'margin_a']
    feats_b = ['conf_b', 'entropy_b', 'margin_b']
    ren_a = {'conf_a': 'confidence', 'entropy_a': 'entropy', 'margin_a': 'margin'}
    ren_b = {'conf_b': 'confidence', 'entropy_b': 'entropy', 'margin_b': 'margin'}

    for seed in SEEDS:
        df_tr, df_te = train_test_split(df_telemetry_full, test_size=0.2, random_state=seed)

        # --- Baselines (No Router) ---
        # 2-Tier No Router (A -> C)
        esc_ac = (df_te['conf_a'] < 0.90).values
        acc_ac = np.mean(np.where(esc_ac, df_te['correct_c'], df_te['correct_a']))
        lat_ac = lat_a + (np.mean(esc_ac) * lat_c)
        all_run_records.append({'Method': 'Pure Threshold (A->C)', 'Accuracy': acc_ac * 100, 'Latency': lat_ac})

        # 3-Tier No Router (A -> B -> C)
        res_3t_base = evaluate_3tier_threshold(df_te, tau1=0.85, tau2=0.90, lat_a=lat_a, lat_b=lat_b, lat_c=lat_c)
        all_run_records.append({'Method': 'Pure Threshold (A->B->C)', 'Accuracy': res_3t_base['Accuracy'] * 100, 'Latency': res_3t_base['Latency (ms)']})

        # Reference: Pure Expert (Model C Only)
        all_run_records.append({'Method': 'Pure Expert (Model C)', 'Accuracy': df_te['correct_c'].mean() * 100, 'Latency': lat_c})

        # --- Meta-Routers (2-Tier & 3-Tier) ---
        target_ac_tr = ((df_tr['correct_a'] == 0) & (df_tr['correct_c'] == 1)).astype(int)

        for r_key, r_meta in ROUTER_REGISTRY.items():
            train_fn = r_meta['train_fn']
            ovh = ROUTER_SINGLE_SAMPLE_OVERHEAD.get(r_key, 0.0)

            # --- 2-Tier: A -> C ---
            r_2t = train_fn(df_tr[feats_a].rename(columns=ren_a), target_ac_tr, lat_c, seed=seed)
            p_2t = r_2t.predict_proba(df_te[feats_a].rename(columns=ren_a))[:, 1]
            esc_2t = (p_2t >= 0.5)
            acc_2t = np.mean(np.where(esc_2t, df_te['correct_c'], df_te['correct_a']))
            lat_2t = lat_a + ovh + (np.mean(esc_2t) * lat_c)
            all_run_records.append({'Method': f"{r_meta['label']} (A->C)", 'Accuracy': acc_2t * 100, 'Latency': lat_2t})

            # --- 3-Tier: A -> B -> C ---
            r_ab = train_fn(df_tr[feats_a].rename(columns=ren_a), df_tr['target_ab'], lat_b, seed=seed)
            r_bc = train_fn(df_tr[feats_b].rename(columns=ren_b), df_tr['target_bc'], lat_c, seed=seed)

            p_ab = r_ab.predict_proba(df_te[feats_a].rename(columns=ren_a))[:, 1]
            p_bc = r_bc.predict_proba(df_te[feats_b].rename(columns=ren_b))[:, 1]

            esc_b = (p_ab >= 0.5)
            esc_c = esc_b & (p_bc >= 0.5)
            exit_b_mask = esc_b & (~(p_bc >= 0.5))
            exit_a_mask = ~esc_b

            acc_3t = np.mean(
                (df_te['correct_a'].values & exit_a_mask) +
                (df_te['correct_b'].values & exit_b_mask) +
                (df_te['correct_c'].values & esc_c)
            )
            lat_3t = lat_a + ovh + (np.mean(esc_b) * (lat_b + ovh)) + (np.mean(esc_c) * lat_c)
            all_run_records.append({'Method': f"{r_meta['label']} (A->B->C)", 'Accuracy': acc_3t * 100, 'Latency': lat_3t})

df_all = pd.DataFrame(all_run_records)
summary = df_all.groupby('Method').agg({
    'Accuracy': ['mean', 'std'],
    'Latency': ['mean', 'std']
})

print("=== Summary Performance Across All Seeds & Datasets ===")
for method in summary.index:
    acc_m = summary.loc[method, ('Accuracy', 'mean')]
    acc_s = summary.loc[method, ('Accuracy', 'std')]
    print(f"{method:<30} Accuracy : {acc_m:.2f} +/- {acc_s:.2f} %")

print("\n=== End-to-End Cascade Latency ===")
for method in summary.index:
    lat_m = summary.loc[method, ('Latency', 'mean')]
    lat_s = summary.loc[method, ('Latency', 'std')]
    print(f"{method:<30} Cascade Latency : {lat_m:.4f} +/- {lat_s:.4f} ms")

print("\n--- Router overhead alone (ablation row) ---")
for r_key, ovh in ROUTER_SINGLE_SAMPLE_OVERHEAD.items():
    label = ROUTER_REGISTRY[r_key]['label']
    print(f"{label:<30} Overhead Alone : {ovh:.4f} ms/sample")

## 6. Evaluation and Naive Baseline


In [ ]:
def naive_threshold_cascade(raw_df, tau, LATENCY_A, LATENCY_C):
    route = (raw_df['confidence'] < tau).astype(int)
    latency = LATENCY_A + route * LATENCY_C
    correct = np.where(route == 1, raw_df['correct_c'], raw_df['correct_a'])
    return {
        'avg_latency': latency.mean(),
        'accuracy': correct.mean() * 100,
    }

def evaluate_router_cascade(router, X_test, raw_test, LATENCY_A, LATENCY_C, theta=0.20):
    n = len(raw_test)
    router_probs = router.predict_proba(X_test)[:, 1]
    decisions = np.where(router_probs >= theta, 1, 0)

    router_overhead_mean, _ = measure_router_overhead_per_sample(router, X_test.head(200))

    latency_total = 0.0
    correct = 0
    for i in range(n):
        latency_total += LATENCY_A + router_overhead_mean
        if raw_test.loc[i, 'confidence'] >= EARLY_EXIT_THRESHOLD:
            correct += raw_test.loc[i, 'correct_a']
        elif decisions[i] == 1:
            latency_total += LATENCY_C
            correct += raw_test.loc[i, 'correct_c']
        else:
            correct += raw_test.loc[i, 'correct_a']

    return {'avg_latency': latency_total / n, 'accuracy': (correct / n) * 100, 'router_overhead': router_overhead_mean}


## 7. Multi-Seed Replication: All Router Families


In [ ]:
def run_seed_trial_comparison(seed, n_samples, chosen_theta, dataset_name, models, transform, latency_a, latency_c, compute_time_acc):
    torch.manual_seed(seed)
    val_dataset_current = get_dataset(dataset_name, transform)
    idx = torch.randperm(len(val_dataset_current))[:n_samples]
    subset = torch.utils.data.Subset(val_dataset_current, idx)
    loader = torch.utils.data.DataLoader(subset, batch_size=32, shuffle=False)

    model_a, _, model_c = models
    df = extract_telemetry(loader, model_a, model_c)
    X = df[['confidence', 'entropy', 'margin']]
    y = df['target_route']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
    raw_test = df.loc[X_test.index].reset_index(drop=True)
    X_test_reset = X_test.reset_index(drop=True)

    acc_expert = raw_test['correct_c'].mean() * 100
    naive_res = naive_threshold_cascade(raw_test, tau=0.80, LATENCY_A=latency_a, LATENCY_C=latency_c)

    result = {
        'dataset': dataset_name,
        'seed': seed,
        'expert_accuracy': acc_expert,
        'expert_latency': latency_c,
        'naive_accuracy': naive_res['accuracy'],
        'naive_latency': naive_res['avg_latency'],
    }

    for name, spec in ROUTER_REGISTRY.items():
        t_start = time.perf_counter()
        router = spec['train_fn'](X_train, y_train, latency_c, seed=seed)
        res = evaluate_router_cascade(router, X_test_reset, raw_test, latency_a, latency_c, theta=chosen_theta)
        t_end = time.perf_counter()

        compute_time_acc[name] = compute_time_acc.get(name, 0.0) + (t_end - t_start)
        result[f'{name}_accuracy'] = res['accuracy']
        result[f'{name}_latency'] = res['avg_latency']
        result[f'{name}_overhead'] = res['router_overhead']

    return result

SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]
CHOSEN_THETA = 0.20
N_SAMPLES_PER_SEED = 4000
DATASETS_TO_EVALUATE = ['CIFAR-100', 'CIFAR-10']

ROUTER_COMPUTE_TIME_SEC = {}
all_trial_results = []

PIPELINE_START = time.perf_counter()
for dataset_name in DATASETS_TO_EVALUATE:
    print(f"\n=== Running multi-router trial for dataset: {dataset_name} ===")
    current_transform = DATASETS[dataset_name]['transform']
    current_models = load_models_for_dataset(dataset_name)

    dataset_trial_results = [
        run_seed_trial_comparison(s, N_SAMPLES_PER_SEED, CHOSEN_THETA, dataset_name, current_models,
                                   current_transform, LATENCY_A, LATENCY_C, ROUTER_COMPUTE_TIME_SEC)
        for s in SEEDS
    ]
    all_trial_results.extend(dataset_trial_results)
PIPELINE_END = time.perf_counter()
TOTAL_PIPELINE_WALLCLOCK_SEC = PIPELINE_END - PIPELINE_START

trial_df = pd.DataFrame(all_trial_results)
print("\nAll multi-seed trials completed successfully.")
print(trial_df.head())


## 8. Summary Performance


In [ ]:
print("=== Summary Performance Across All Seeds & Datasets ===")
for name, spec in ROUTER_REGISTRY.items():
    print(f"{spec['label']:<20s} Accuracy : {trial_df[f'{name}_accuracy'].mean():.2f} +/- {trial_df[f'{name}_accuracy'].std():.2f} %")
print(f"{'Pure Expert':<20s} Accuracy : {trial_df['expert_accuracy'].mean():.2f} +/- {trial_df['expert_accuracy'].std():.2f} %")
print(f"{'Naive Baseline':<20s} Accuracy : {trial_df['naive_accuracy'].mean():.2f} +/- {trial_df['naive_accuracy'].std():.2f} %")
print()
for name, spec in ROUTER_REGISTRY.items():
    print(f"{spec['label']:<20s} Cascade Latency : {trial_df[f'{name}_latency'].mean():.4f} +/- {trial_df[f'{name}_latency'].std():.4f} ms")
print(f"{'Pure Expert':<20s} Cascade Latency : {trial_df['expert_latency'].mean():.4f} ms")
print()
print("--- Router overhead alone (ablation row) ---")
for name, spec in ROUTER_REGISTRY.items():
    print(f"{spec['label']:<20s} Overhead Alone : {trial_df[f'{name}_overhead'].mean():.4f} +/- {trial_df[f'{name}_overhead'].std():.4f} ms/sample")


## 9. Paired Statistical Significance Testing

Tests whether XGBoost's performance differs significantly from Random Forest.

In [ ]:
def paired_report(a, b, label_a, label_b, unit='ms'):
    diffs = np.array(a) - np.array(b)
    t_stat, p_val = stats.ttest_rel(a, b)
    try:
        w_stat, w_p = stats.wilcoxon(a, b)
    except ValueError:
        w_stat, w_p = np.nan, np.nan
    print(f"  Mean Diff ({label_a} - {label_b}): {diffs.mean():+.3f} {unit}, Std: {diffs.std():.3f} {unit}")
    print(f"  Paired t-test        : t = {t_stat:.3f}, p = {p_val:.4f}")
    print(f"  Wilcoxon signed-rank : p = {w_p:.4f}")
    print()

router_names = list(ROUTER_REGISTRY.keys())
for dataset_name in trial_df['dataset'].unique():
    print(f"--- Paired Tests for {dataset_name} ---")
    ds_df = trial_df[trial_df['dataset'] == dataset_name]

    for name_a, name_b in itertools.combinations(router_names, 2):
        label_a, label_b = ROUTER_REGISTRY[name_a]['label'], ROUTER_REGISTRY[name_b]['label']

        print(f"{label_a} vs. {label_b} (Accuracy):")
        paired_report(ds_df[f'{name_a}_accuracy'], ds_df[f'{name_b}_accuracy'], label_a, label_b, unit='pp')

        print(f"{label_a} vs. {label_b} (Cascade Latency, incl. overhead):")
        paired_report(ds_df[f'{name_a}_latency'], ds_df[f'{name_b}_latency'], f'{label_a} Latency', f'{label_b} Latency', unit='ms')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Boost font sizes for small figure rendering
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
})

pairs = [
    'XGB vs. RF', 'XGB vs. LogReg', 'XGB vs. DT', 'XGB vs. GA',
    'RF vs. LogReg', 'RF vs. DT', 'RF vs. GA',
    'LogReg vs. DT', 'LogReg vs. GA', 'DT vs. GA'
]

# CIFAR-100 Data
c100_acc_mean = np.array([-0.047, 0.906, 0.453, 4.359, 0.953, 0.500, 4.406, -0.453, 3.453, 3.906])
c100_acc_std  = np.array([0.324, 0.637, 0.552, 0.751, 0.707, 0.512, 0.990, 0.734, 1.059, 1.118])
c100_lat_mean = np.array([-3.040, 0.802, 0.363, 2.349, 3.842, 3.403, 5.389, -0.439, 1.547, 1.986])
c100_lat_std  = np.array([0.126, 0.166, 0.390, 0.739, 0.131, 0.340, 0.729, 0.373, 0.712, 0.760])

# CIFAR-10 Data
c10_acc_mean = np.array([0.000, 0.156, 0.031, 0.281, 0.156, 0.031, 0.281, -0.125, 0.125, 0.250])
c10_acc_std  = np.array([0.000, 0.185, 0.083, 0.263, 0.185, 0.083, 0.263, 0.225, 0.125, 0.306])
c10_lat_mean = np.array([-2.857, 0.266, 0.266, 0.548, 3.123, 3.123, 3.405, 0.000, 0.283, 0.283])
c10_lat_std  = np.array([0.078, 0.013, 0.016, 0.068, 0.081, 0.084, 0.094, 0.027, 0.062, 0.076])

y_pos = np.arange(len(pairs))[::-1]

fig, axes = plt.subplots(2, 2, figsize=(9, 5), sharey=True)

def plot_forest(ax, mean, std, title, xlabel, color):
    ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.6)
    ax.errorbar(mean, y_pos, xerr=std, fmt='o', color=color, ecolor=color,
                elinewidth=2, capsize=3.5, capthick=1.5, markersize=5, zorder=3)
    ax.set_title(title, fontweight='bold', pad=6)
    ax.set_xlabel(xlabel, labelpad=4)
    ax.grid(axis='x', linestyle=':', alpha=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(pairs)

# CIFAR-100
plot_forest(axes[0, 0], c100_acc_mean, c100_acc_std, 'CIFAR-100: Accuracy', r'$\Delta$ Acc (pp, $\pm 1\sigma$)', '#1f77b4')
plot_forest(axes[0, 1], c100_lat_mean, c100_lat_std, 'CIFAR-100: Latency', r'$\Delta$ Lat (ms, $\pm 1\sigma$)', '#d62728')

# CIFAR-10
plot_forest(axes[1, 0], c10_acc_mean, c10_acc_std, 'CIFAR-10: Accuracy', r'$\Delta$ Acc (pp, $\pm 1\sigma$)', '#1f77b4')
plot_forest(axes[1, 1], c10_lat_mean, c10_lat_std, 'CIFAR-10: Latency', r'$\Delta$ Lat (ms, $\pm 1\sigma$)', '#d62728')

plt.tight_layout()
plt.savefig('stat-sig.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Visualizing Pareto Frontiers


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for name, spec in ROUTER_REGISTRY.items():
    ax.errorbar(trial_df[f'{name}_latency'].mean(), trial_df[f'{name}_accuracy'].mean(),
                xerr=trial_df[f'{name}_latency'].std(), yerr=trial_df[f'{name}_accuracy'].std(),
                fmt=spec['marker'], color=spec['color'], capsize=5, markersize=10,
                label=f"{spec['label']} (mean +/- std)")

ax.errorbar(trial_df['naive_latency'].mean(), trial_df['naive_accuracy'].mean(),
            xerr=trial_df['naive_latency'].std(), yerr=trial_df['naive_accuracy'].std(),
            fmt='s', color='tab:green', capsize=5, markersize=10, label='Naive Threshold Baseline')

ax.errorbar(trial_df['expert_latency'].mean(), trial_df['expert_accuracy'].mean(),
            xerr=0, yerr=trial_df['expert_accuracy'].std(),
            fmt='*', color='tab:red', capsize=5, markersize=15, label='Pure Expert (Model C)')

ax.set_xlabel('Average Latency (ms/sample)', fontsize=11)
ax.set_ylabel('End-to-End Accuracy (%)', fontsize=11)
ax.set_title('Pareto Comparison: Meta-Router Family vs. Cascade Latency/Accuracy', fontsize=13, fontweight='bold')
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.savefig('router_family_pareto.png', dpi=150)
plt.show()


## 11. Total Computation Time

Per-sample latency (above) measures production-time cost. This section reports the **total wall-clock compute time** for the experiment pipeline itself -- training + evaluating every router family, across all 8 seeds and both datasets -- which is a separate number your report should include alongside the per-sample figures.

In [ ]:
print(f"Total pipeline wall-clock time (all datasets, all seeds, all routers): {TOTAL_PIPELINE_WALLCLOCK_SEC:.2f} s "
      f"({TOTAL_PIPELINE_WALLCLOCK_SEC/60:.2f} min)")
print()
print("--- Cumulative train+eval compute time per router family (all seeds x all datasets) ---")
for name, spec in ROUTER_REGISTRY.items():
    total_s = ROUTER_COMPUTE_TIME_SEC.get(name, float('nan'))
    share = 100 * total_s / sum(ROUTER_COMPUTE_TIME_SEC.values())
    print(f"{spec['label']:<20s} : {total_s:.2f} s  ({share:.1f}% of all router compute time)")


In [ ]:
seed = 1
dataset_name = 'CIFAR-10'
torch.manual_seed(seed)
transform = DATASETS[dataset_name]['transform']
models = load_models_for_dataset(dataset_name)
val_dataset_current = get_dataset(dataset_name, transform)
idx = torch.randperm(len(val_dataset_current))[:N_SAMPLES_PER_SEED]
subset = torch.utils.data.Subset(val_dataset_current, idx)
loader = torch.utils.data.DataLoader(subset, batch_size=32, shuffle=False)

model_a, _, model_c = models
df = extract_telemetry(loader, model_a, model_c)
X = df[['confidence', 'entropy', 'margin']]
y = df['target_route']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
X_test_reset = X_test.reset_index(drop=True)

xgb_router = train_xgb_router(X_train, y_train, LATENCY_C, seed=seed)
rf_router = train_rf_router(X_train, y_train, LATENCY_C, seed=seed)

xgb_probs = xgb_router.predict_proba(X_test_reset)[:, 1]
rf_probs = rf_router.predict_proba(X_test_reset)[:, 1]
xgb_decisions = (xgb_probs >= CHOSEN_THETA).astype(int)
rf_decisions = (rf_probs >= CHOSEN_THETA).astype(int)

agreement = (xgb_decisions == rf_decisions).mean() * 100
print(f"XGBoost/RF decision agreement on {dataset_name}, seed {seed}: {agreement:.2f}%")
print(f"XGBoost escalation rate: {xgb_decisions.mean()*100:.2f}%")
print(f"RF escalation rate: {rf_decisions.mean()*100:.2f}%")

# Where they DO disagree, do the disagreements roughly cancel out in accuracy?
disagree_mask = xgb_decisions != rf_decisions
print(f"Samples where they disagree: {disagree_mask.sum()} / {len(disagree_mask)}")

raw_test = df.loc[X_test.index].reset_index(drop=True)
disagreement_rows = raw_test.loc[disagree_mask]
print((disagreement_rows['correct_a'] == disagreement_rows['correct_c']).mean())

In [ ]:
plt.rcParams.update({
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 8.5,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 6.5,
    'lines.linewidth': 1.0,
    'lines.markersize': 3,
})

router_names = list(ROUTER_REGISTRY.keys())
labels = [ROUTER_REGISTRY[n]['label'] for n in router_names]
datasets = trial_df['dataset'].unique()
n_ds = len(datasets)

fig, axes = plt.subplots(n_ds * 2, 1, figsize=(3.5, 2.3 * n_ds * 2), sharex=False)
if n_ds == 1:
    axes = np.array([axes])

for row, dataset_name in enumerate(datasets):
    ds_df = trial_df[trial_df['dataset'] == dataset_name]
    acc_data = [ds_df[f'{n}_accuracy'].values for n in router_names]
    lat_data = [ds_df[f'{n}_latency'].values for n in router_names]

    # Accuracy Plot
    ax_acc = axes[row * 2]
    ax_acc.boxplot(acc_data, labels=labels, showmeans=True, widths=0.5)
    for i, d in enumerate(acc_data):
        jitter_x = np.random.normal(i + 1, 0.04, size=len(d))
        ax_acc.scatter(jitter_x, d, alpha=0.6, s=8, color='black', zorder=3)
    ax_acc.set_title(f'{dataset_name}: Accuracy Distribution (n=8 seeds)')
    ax_acc.set_ylabel('Accuracy (%)')
    ax_acc.tick_params(axis='x', rotation=30)
    ax_acc.grid(True, linestyle=':', alpha=0.5)

    # Latency Plot
    ax_lat = axes[row * 2 + 1]
    ax_lat.boxplot(lat_data, labels=labels, showmeans=True, widths=0.5)
    for i, d in enumerate(lat_data):
        jitter_x = np.random.normal(i + 1, 0.04, size=len(d))
        ax_lat.scatter(jitter_x, d, alpha=0.6, s=8, color='black', zorder=3)
    ax_lat.axhline(ds_df['expert_latency'].mean(), color='red', linestyle='--',
                   linewidth=1.0, label='Pure Expert')
    ax_lat.set_title(f'{dataset_name}: Latency Distribution (n=8 seeds)')
    ax_lat.set_ylabel('Latency (ms)')
    ax_lat.tick_params(axis='x', rotation=30)
    ax_lat.grid(True, linestyle=':', alpha=0.5)
    ax_lat.legend(loc='upper right', framealpha=0.8)

plt.tight_layout()
plt.savefig('fig_raw_distributions_single_col.pdf', bbox_inches='tight', facecolor='white')
plt.savefig('fig_raw_distributions_single_col.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()